In [1]:
from langchain_openai import ChatOpenAI
from langchain_community.document_loaders import PyPDFLoader, JSONLoader, CSVLoader, WebBaseLoader, ImageCaptionLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter, TextSplitter
from langchain_openai import OpenAIEmbeddings
from langchain_community.vectorstores import FAISS
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import StrOutputParser
from glob import glob

USER_AGENT environment variable not set, consider setting it to identify your requests.


In [4]:
csvloader = CSVLoader(file_path='data/test.csv')
document = csvloader.load() 
print( document )

[Document(metadata={'source': 'data/test.csv', 'row': 0}, page_content='name: Alice\nage: 30\njob: Engineer'), Document(metadata={'source': 'data/test.csv', 'row': 1}, page_content='name: Bob\nage: 25\njob: Designer'), Document(metadata={'source': 'data/test.csv', 'row': 2}, page_content='name: Charlie\nage: 35\njob: Manager')]


In [6]:
# 문서 분할
embedding = OpenAIEmbeddings(model='text-embedding-ada-002')
vdb = FAISS.from_documents( document, embedding)

In [ ]:
# 프롬프트 템플릿 정의
template = """질문에 대해 아래의 제공된 문맥(context)만을 사용하여 답변하세요. 
답을 모른다면 모른다고 말하고 답변을 지어내지 마세요. 
최대한 세 문장 내로 간결하게 답변하세요.

Question: {question} 
Context: {context} 
Answer:"""
prompt = ChatPromptTemplate.from_template(template)

llm = ChatOpenAI( model='gpt-4o-mini', temperature=0)
retriever = vdb.as_retriever() # vdb.as_retriever(k=4) 

def format_docs(docs):
    # print( "문서:", docs.page_content)
    return "\n".join([doc.page_content for doc in docs])

# 유사도 검색
qa = ( 
        {"context": retriever | format_docs, "question": RunnablePassthrough()} # 유사도 검색에서 반환된 문서들을 하나의 문자열로 변환
        | prompt  
        | llm  
        | StrOutputParser() 
    )
query = "엔지니어 이름은? " # question
result = qa.invoke( query )
print( result )

엔지니어의 이름은 Alice입니다.
